In [ ]:
%load_ext autoreload
%autoreload 2
import torch
from dotenv import load_dotenv
from accelerate import Accelerator
from constant import *
from GeminiModel import GeminiModel
from TrainStrategy import TrainStrategy
from LlmSatdOutputLabelConverter import LlmSatdOutputLabelConverter

In [ ]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
prompt_template = PromptTemplate(
    name="Manually Crafted",
    definition="You are a Code Analysis Expert specialized in detecting Self-Admitted Technical Debt (SATD) in Java test code comments. SATD refers to comments where developers acknowledge that the current test implementation is incomplete, suboptimal, or relies on a compromise that should be addressed in the future. These admissions often appear as markers such as TODO or FIXME, or as notes about unresolved issues, temporary fixes, workarounds, hacks, performance limitations, use of deprecated APIs, unsupported features, poor design choices, skipped tests, or uncertain functionality. However, comments that only describe expected behavior, provide instructions, or reference external issues (e.g., JIRA ID) are not SATD unless there is additional information indicating the need for future improvement.",
    instruction="Think step by step and assign the label of **SATD** or **Not-SATD** for each given test code comment.",
    n_shot_template="Comment: {{ text }}",
    n_shot_answer_template="""{% if cot -%}
        Answer: {{ cot }} The answer is **{{ label }}**.
        {% endif -%}""",
    line_m_before=3,
    line_n_after=3
)
output_label_converter = LlmSatdOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)

# Detect with Gemini 2.0 Flash N-Shots

In [ ]:
gemini_2_flash_detection_model = GeminiModel('detect', 'models/gemini-2.0-flash', output_label_converter)
gemini_2_flash_detection_model.fit(detect_n_shot_dataset)
gemini_2_flash_detection_model.predict(detect_test_dataset.select(range(1)), DETECT_DATASET_NAME, prompt_template, TrainStrategy.N_SHOT_TOP, 10, verbose=True)